In [2]:

#* Definimos la carpeta donde se almacenarán los datos persistentes de ChromaDB

import os
import chromadb
from chromadb.utils import embedding_functions

PERSIST_DIR = "./db_chroma_andino"
os.makedirs(PERSIST_DIR, exist_ok=True)

print(f"Carpeta de persistencia: {os.path.abspath(PERSIST_DIR)}")

Carpeta de persistencia: d:\nalvarez\100_cursos\bases_vectoriales\modulos\Chroma_bd\db_chroma_andino


In [3]:

#* Creamos PersistentClient y obtenemos (o creamos) la colección. Usaremos la función de embeddings por defecto de Chroma

client = chromadb.PersistentClient(path=PERSIST_DIR)

default_ef = embedding_functions.DefaultEmbeddingFunction()

COLLECTION_NAME = "banco_andino_v1"

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=default_ef
)

print(f"Colección lista: {collection.name}")

Colección lista: banco_andino_v1


In [4]:

#* Ahora con la colección creada, podemos añadir documentos para su posterior búsqueda y recuperación

documentos_banco_andino = [
    {
        "id": "consumo_clasico",
        "texto": """
Crédito de consumo clásico dirigido a personas asalariadas con al menos 12 meses de estabilidad laboral.
Monto: USD 1,000 a 10,000. Plazo: 6 a 36 meses.
Requisitos: comprobante de ingresos, sin mora mayor a 30 días en los últimos 12 meses.
La cuota mensual no debe superar el 35% del ingreso neto del cliente.
""",
    },
    {
        "id": "nomina_convenio",
        "texto": """
Crédito con descuento por nómina para empleados de empresas con convenio vigente con el banco.
No requiere codeudor. Antigüedad mínima: 6 meses.
Monto máximo: hasta 8 veces el salario neto mensual.
El pago se realiza vía deducción automática en planilla.
""",
    },
    {
        "id": "hipotecario_primera_vivienda",
        "texto": """
Crédito hipotecario para primera vivienda.
Financia hasta el 80% del valor de tasación del inmueble. Plazo hasta 20 años.
Requisitos: enganche mínimo del 20%, sin registros negativos en los últimos 24 meses.
Las obligaciones mensuales totales (incluida la hipoteca) no deben superar el 40% del ingreso familiar neto.
""",
    },
    {
        "id": "pyme_capital_trabajo",
        "texto": """
Crédito PYME Capital de Trabajo para empresas con al menos 2 años de operación formal.
Montos desde USD 5,000 hasta USD 200,000. Plazo hasta 24 meses.
Requiere estados financieros, flujo de caja proyectado y, según el riesgo, garantías reales o fideicomisos.
""",
    },
    {
        "id": "politica_riesgo_general",
        "texto": """
Política general de riesgo de crédito.
Se evalúan estabilidad laboral o del negocio, nivel de endeudamiento, score interno y externo,
y comportamiento histórico con el banco.
Solicitudes con endeudamiento total superior al 45% del ingreso neto se consideran solo de forma excepcional.
No se aprueban créditos con moras activas mayores a 90 días al momento de la evaluación.
""",
    },
]

In [5]:

#* Convertimos la lista de datos al formato que requiere ChromaDB

docs_banco_andino = {
    "ids": [doc['id'] for doc in documentos_banco_andino],
    "documentos": [doc['texto'] for doc in documentos_banco_andino]
}

print(f"Documentos preparados: {len(docs_banco_andino['ids'])}")

Documentos preparados: 5


In [6]:

#* Para la inserción de los datos en la colección usamos upsert (de esta forma si ya existen los ids, se actualizan)
collection.upsert(
    ids=docs_banco_andino["ids"],
    documents=docs_banco_andino["documentos"],
    metadatas=docs_banco_andino.get("metadatas")
)

print(f"Documentos insertados/actualizados en la colección '{collection.name}'")

Documentos insertados/actualizados en la colección 'banco_andino_v1'


In [9]:

#* Generamos una función para mejorar la respuesta de las consultas
def imprimir_resultados(results, titulo="Resultados"):
    print("="*90)
    print(titulo)
    print("="*90)
    
    ids = results.get("ids", [[]])[0]
    documents = results.get("documents", [[]])[0]
    dists = results.get("distances", [[]])[0]
    
    for i, (rid, rdoc, rdist) in enumerate(zip(ids, documents, dists), start=1):
        print(f"Top {i} | ID: {rid:<28} | Distancia: {rdist:.4f}")
        snippet = (rdoc[:180] + '...') if len(rdoc) > 180 else rdoc
        print(f"         | Documento: {snippet}")
        print("-"*90)
    
    print("Nota: menor distancia indica mayor similitud.")

In [10]:
consulta_rel = "¿Cuál es el monto máximo que puedo solicitar con descuento por nómina?"
res_rel = collection.query(query_texts=[consulta_rel], n_results=3)
imprimir_resultados(res_rel, titulo="(Persistencia) Consulta relacionada - Monto máximo por nómina")

(Persistencia) Consulta relacionada - Monto máximo por nómina
Top 1 | ID: nomina_convenio              | Distancia: 0.6704
         | Documento: 
Crédito con descuento por nómina para empleados de empresas con convenio vigente con el banco.
No requiere codeudor. Antigüedad mínima: 6 meses.
Monto máximo: hasta 8 veces el sal...
------------------------------------------------------------------------------------------
Top 2 | ID: hipotecario_primera_vivienda | Distancia: 1.0327
         | Documento: 
Crédito hipotecario para primera vivienda.
Financia hasta el 80% del valor de tasación del inmueble. Plazo hasta 20 años.
Requisitos: enganche mínimo del 20%, sin registros negati...
------------------------------------------------------------------------------------------
Top 3 | ID: politica_riesgo_general      | Distancia: 1.0996
         | Documento: 
Política general de riesgo de crédito.
Se evalúan estabilidad laboral o del negocio, nivel de endeudamiento, score interno y externo,
y co